# Pulpermill backend (Kaggle launcher)
Settings: **Accelerator = GPU T4 x2**, **Internet = on**. Add two secrets (Add-ons, Secrets): `NGROK_AUTHTOKEN` and `PULPERMILL_TOKEN` (any long random string; type the same one into the page).
No logic lives here: everything is in `pulpermill/backend/launch.py`. Run the cells in order.

## 1. Install

In [ ]:
import os, subprocess, sys
BRANCH = "main"
REPO = "https://github.com/LacChe/pulpermill"
if not os.path.exists("pulpermill"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO], check=True)
os.chdir("pulpermill")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "pulpermill/backend/requirements.txt", "pyngrok", "huggingface_hub"], check=True)

## 2. llama-cpp-python with CUDA
Uses a prebuilt CUDA wheel (fast). If your earlier notebook had an install line that worked for Gemma 3 on the T4s, use that instead.

In [ ]:
import subprocess, sys
try:
    import llama_cpp
    print("llama-cpp-python", llama_cpp.__version__, "is already installed")
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python",
                        "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu124"])
    if r.returncode:
        env = dict(os.environ, CMAKE_ARGS="-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python", "--no-cache-dir"], env=env, check=True)

## 3. Load the model, start the server, open the tunnel
This cell keeps running; the address is printed when it is ready.

In [ ]:
from kaggle_secrets import UserSecretsClient
from pulpermill.backend.launch import launch
secrets = UserSecretsClient()
launch(token=secrets.get_secret("PULPERMILL_TOKEN"), ngrok_authtoken=secrets.get_secret("NGROK_AUTHTOKEN"))

## Measuring (run once, in a separate session or after stopping cell 3)
Fills in `docs/backend-notes.md`: tokenizer parity, speed, cancellation, memory. Copy the printed Markdown back.

In [ ]:
from dataclasses import replace
from pulpermill.backend.config import Config
from pulpermill.backend.engine import download
from pulpermill.backend.engine.llama import load
from tools.measure_engine import measure_all, memory_by_ctx
model = Config().model
path = download.ensure_model(model)
engine = load(model, path)
measure_all(engine)

## Trials on the real model (for the prompts work)
Runs `tools/run_trials.py`'s chapter loop in this session's engine, with no server. Needs the engine loaded as in the measuring cell. Each call's sampling and token/second record is kept in `rows`.

In [ ]:
from pulpermill.backend.complete import make_complete
from pulpermill.backend.config import Config
from tools import run_trials

rows = []
complete = make_complete(engine, max_tokens=1600, params=Config().prose, record=rows)
# request = <a generate request dict, e.g. from tests/trials>
# chapter, sections = run_trials.run_chapter(request, complete)


In [ ]:
del engine
memory_by_ctx(model, path, [8192, 12288, 16384])